In [1]:
url = 'https://prod-dcd-datasets-public-files-eu-west-1.s3.eu-west-1.amazonaws.com/0899f4de-4ac7-48d3-b317-a0365c88cfa8'
import requests
import pathlib
import zipfile
try:
  import wget
except:
  !pip install wget
  import wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=7fe2fc8b52a02ad8f0007f65c1cda975ebc0acdb233bf0340938526ee24a5d44
  Stored in directory: /root/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [2]:
parent_path = pathlib.Path("data/")
zip_file = parent_path / "data.zip"
images_path = parent_path / "images"
if images_path.exists():
  print("Data is already present.")
else:
  images_path.mkdir(exist_ok=True, parents=True)
  if zip_file.exists():
    print("File already downloaded.")
  else:
    print('Downloading file...')
    wget.download(url, str(zip_file))
    with zipfile.ZipFile(zip_file) as z:
      print('Extracting file...')
      z.extractall(parent_path)
      print("Data fetched successfully.")

Extracting file...
Data fetched successfully.


In [3]:
!pip install split-folders
from splitfolders import ratio

In [4]:
data_path = parent_path / "SignAlphaSet/"
ratio(input=data_path, output=images_path, seed=42, ratio=(0.8, 0.1, 0.1))
test_path = images_path / "test/"
train_path = images_path / "train/"
validation_path = images_path / 'val/'

Copying files: 26000 files [00:04, 5686.21 files/s]


In [5]:
from torchvision.models import EfficientNet_B2_Weights
weights = EfficientNet_B2_Weights.DEFAULT
auto_transform = weights.transforms() # get transform used to train that model
auto_transform

ImageClassification(
    crop_size=[288]
    resize_size=[288]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

In [6]:
from torchvision.datasets import ImageFolder
from torch.utils.data.dataloader import DataLoader
train_data = ImageFolder(train_path, auto_transform)
test_data = ImageFolder(test_path, auto_transform)
val_data = ImageFolder(validation_path, auto_transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
train_loader

In [7]:
try:
  from torchmetrics import Accuracy
except:
  !pip install torchmetrics
  from torchmetrics import Accuracy
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
acc_fn = Accuracy(num_classes=26, task='multiclass').to(device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 59.6 MB/s eta 0:00:00


In [8]:
import torchvision

model = torchvision.models.efficientnet_b2(weights=weights)
model

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 88.1MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [9]:
# from torchsummary import summary
# summary(model=model, input_size=(3, 288, 288), batch_size=32)

In [10]:
for param in model.parameters():
  param.requires_grad = False # freeze all parameters

# summary(model=model, input_size=(3, 288, 288), batch_size=32)

## Checking accuracy on training only classifier

In [11]:
model.classifier = torch.nn.Sequential(torch.nn.Dropout(p=0.3, inplace=True), torch.nn.Linear(in_features=1408, out_features=26, bias=True))
# summary(model=model, input_size=(3, 288, 288), batch_size=32)

In [12]:
from tqdm import tqdm
def train_step(model, dataloader, optimiser, loss_fn, acc_fn, device):
  train_loss = 0
  train_accuracy = 0
  model.train()
  for images, labels in tqdm(dataloader, desc='Training'):
    images, labels = images.to(device), labels.to(device)
    y_probs = torch.softmax(model(images), dim=1)
    y_pred = torch.argmax(y_probs, dim=1)
    loss = loss_fn(y_probs, labels)
    acc = acc_fn(y_pred, labels)

    train_loss += loss.cpu().item()
    train_accuracy += acc.cpu().item()

    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
  return train_loss / len(dataloader), train_accuracy / len(dataloader)


def evaluation_step(model, dataloader, loss_fn, acc_fn, device):
  val_loss = 0
  val_accuracy = 0
  model.eval()
  with torch.inference_mode():
    for images, labels in tqdm(dataloader, desc='Validating'):
      images, labels = images.to(device), labels.to(device)
      y_probs = torch.softmax(model(images), dim=1)
      y_pred = torch.argmax(y_probs, dim=1)
      loss = loss_fn(y_probs, labels)
      acc = acc_fn(y_pred, labels)

      val_loss += loss.cpu().item()
      val_accuracy += acc.cpu().item()

  return val_loss / len(dataloader), val_accuracy / len(dataloader)

In [13]:
class EarlyStopping:
  def __init__(self, patience=5):
    super().__init__()
    self.patience = 5
    self.counter = 0
    self.best_loss = None

  def stop(self, loss):
    if self.best_loss is None:
      self.best_loss = loss
      return False
    else:
      if loss >= self.best_loss:
        if self.counter >=self.patience:
          print("Early Stopping.")
          return True
        else:
          self.counter += 1
          return False
      return False

In [14]:
def train_model(model, train_loader, val_loader, optimiser, loss_fn, acc_fn, device, epochs):
  train_losses = []
  val_losses = []
  val_accs = []
  train_accs = []
  early_stopper = EarlyStopping(5)
  for epoch in range(1, epochs + 1):
    print(f"Epoch: {epoch}/{epochs}")
    train_loss, train_acc = train_step(model=model, dataloader=train_loader, optimiser=optimiser, loss_fn=loss_fn, acc_fn=acc_fn, device=device)
    val_loss, val_acc = evaluation_step(model=model, dataloader=val_loader, loss_fn=loss_fn, acc_fn=acc_fn, device=device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    print(f"Train Accuracy: {train_acc} | Validation Accuracy: {val_acc} | Training Loss: {train_loss} | Validation Loss: {val_loss}")
    if early_stopper.stop(val_loss):
      break
  return {"train_losses": train_losses, "val_losses": val_losses, "train_accuracies":train_accs, "val_accuracies":val_accs}

In [15]:
loss_fn = torch.nn.CrossEntropyLoss()
model=model.to(device)
optimiser = torch.optim.Adam(model.parameters(), lr=0.001)

# train for 5 epochs
torch.manual_seed(42)
torch.cuda.manual_seed(42)
train_loss, val_loss, train_acc, val_acc = train_model(model=model, train_loader=train_loader, val_loader=val_loader, acc_fn=acc_fn, loss_fn=loss_fn, device=device, optimiser=optimiser, epochs=5)

Epoch: 1/5


Validating: 100%|██████████| 41/41 [00:21<00:00,  1.88it/s]


Train Accuracy: 0.9325961538461538 | Validation Accuracy: 1.0 | Training Loss: 2.5146333738473747 | Validation Loss: 2.330894691188161
Epoch: 2/5


Validating: 100%|██████████| 41/41 [00:22<00:00,  1.83it/s]


Train Accuracy: 0.9985576923076923 | Validation Accuracy: 1.0 | Training Loss: 2.339785291965191 | Validation Loss: 2.3246577890907845
Epoch: 3/5


...

Train Accuracy: 0.9991826923076923 | Validation Accuracy: 1.0 | Training Loss: 2.33135709469135 | Validation Loss: 2.3234749538142507
Epoch: 4/5


Validating: 100%|██████████| 41/41 [00:21<00:00,  1.90it/s]


Train Accuracy: 0.999375 | Validation Accuracy: 1.0 | Training Loss: 2.3285306673783523 | Validation Loss: 2.322916263487281
Epoch: 5/5


Validating: 100%|██████████| 41/41 [00:22<00:00,  1.85it/s]

Train Accuracy: 0.999375 | Validation Accuracy: 1.0 | Training Loss: 2.3268136596679687 | Validation Loss: 2.3225993005240837


In [16]:
test_loss, test_accuracy = evaluation_step(model=model, loss_fn=loss_fn, dataloader=test_loader, acc_fn=acc_fn, device=device)
print(f"Test loss is {test_loss} & Accuracy is {test_accuracy}")

Validating: 100%|██████████| 41/41 [00:22<00:00,  1.84it/s]

Test loss is 2.322702745111977 & Accuracy is 1.0


In [17]:
print('Saving model..')
torch.save(model.state_dict(), "ASL_CNN.pth")
print("Saved")

Saving model..
Saved
